# IEQ Hyperparameter Tuning - 5-Fold Cross Validation

This notebook tunes model hyperparameters using 5-fold cross-validation and then evaluates the tuned models with another 5-fold cross-validation pass. The tuning objective is macro F1, which is more informative than raw accuracy for these imbalanced three-class targets.


## 1. Imports and Configuration


In [1]:

from __future__ import annotations

import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATASET_RELATIVE_PATH = Path("02_Datasets") / "model_ready" / "ieq_model_dataset.xlsx"
DATASET_FILE_NAME = "ieq_model_dataset.xlsx"


def find_dataset_path() -> Path:
    search_roots = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    for root in search_roots:
        candidates.extend(
            [
                root / DATASET_RELATIVE_PATH,
                root / "Prism" / DATASET_RELATIVE_PATH,
                root / DATASET_FILE_NAME,
            ]
        )
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find ieq_model_dataset.xlsx. Keep the Prism folder structure, "
        "or place ieq_model_dataset.xlsx in the current notebook folder."
    )


DATASET_PATH = find_dataset_path()
if DATASET_PATH.parent.name == "model_ready" and DATASET_PATH.parent.parent.name == "02_Datasets":
    PROJECT_ROOT = DATASET_PATH.parent.parent.parent
    OUTPUT_DIR = PROJECT_ROOT / "03_Code" / "ieq_paper" / "02_outputs"
else:
    PROJECT_ROOT = Path.cwd()
    OUTPUT_DIR = PROJECT_ROOT / "ieq_model_outputs"

PROJECT_TOOLS_DIR = PROJECT_ROOT / "03_Code" / "_project_tools"
if PROJECT_TOOLS_DIR.exists() and str(PROJECT_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_TOOLS_DIR))

try:
    from paper_style import COLORS as PAPER_COLORS, apply_paper_style, save_figure, style_axes
except ModuleNotFoundError:
    PAPER_COLORS = {
        "text": "#111827",
        "border": "#1F2933",
        "axis": "#AAB2B8",
        "grid": "#D9DEE3",
        "best": "#2A9D8F",
        "medium": "#E9C46A",
        "dark_blue": "#457B9D",
        "secondary_green": "#5A9E6F",
        "purple": "#7C6EE6",
    }

    def apply_paper_style() -> None:
        plt.rcParams.update(
            {
                "figure.facecolor": "white",
                "axes.facecolor": "white",
                "savefig.facecolor": "white",
                "savefig.bbox": "tight",
                "savefig.dpi": 300,
                "font.family": "sans-serif",
                "font.size": 10.5,
                "axes.labelcolor": PAPER_COLORS["text"],
                "axes.edgecolor": PAPER_COLORS["axis"],
                "axes.grid": True,
                "axes.axisbelow": True,
                "grid.color": PAPER_COLORS["grid"],
                "legend.frameon": False,
            }
        )

    def style_axes(ax, grid_axis: str = "x") -> None:
        ax.grid(axis=grid_axis, color=PAPER_COLORS["grid"], alpha=0.85, linewidth=0.8)
        ax.set_axisbelow(True)
        for spine in ("top", "right", "left"):
            ax.spines[spine].set_visible(False)
        ax.spines["bottom"].set_color(PAPER_COLORS["axis"])
        ax.tick_params(axis="y", length=0)

    def save_figure(fig, output_path: Path | str, dpi: int = 300) -> None:
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path.with_suffix(".png"), dpi=dpi, bbox_inches="tight")
        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")

apply_paper_style()

SEED = 20260507
TUNING_SPLITS = 5
EVALUATION_SPLITS = 5
SCORING = "f1_macro"
CLASS_ORDER = ["dissatisfied", "neutral", "satisfied"]
CLASS_TO_NUMBER = {label: index for index, label in enumerate(CLASS_ORDER)}
CLASS_DISPLAY_LABELS = ["Dissatisfied", "Neutral", "Satisfied"]

MODEL_ORDER = ["Logistic regression", "Random Forest", "Extra Trees", "Gradient boosting", "ANN / MLP"]
MODEL_LABELS = {
    "Logistic regression": "Logistic\nregression",
    "Random Forest": "Random\nForest",
    "Extra Trees": "Extra\nTrees",
    "Gradient boosting": "Gradient\nboosting",
    "ANN / MLP": "ANN /\nMLP",
}
MODEL_COLORS = {
    "Logistic regression": PAPER_COLORS["dark_blue"],
    "Random Forest": PAPER_COLORS["secondary_green"],
    "Extra Trees": PAPER_COLORS["best"],
    "Gradient boosting": PAPER_COLORS["medium"],
    "ANN / MLP": PAPER_COLORS["purple"],
}
CONFUSION_CMAP = LinearSegmentedColormap.from_list(
    "paper_confusion",
    ["#FFFFFF", "#D9DEE3", PAPER_COLORS["dark_blue"]],
)


def model_value_label_color(model: str) -> str:
    return PAPER_COLORS["text"] if model == "Gradient boosting" else "white"

TARGET = "IEQ satisfaction 3-class"
DROP_FROM_FEATURES = ["IEQ satisfaction 3-class", "TimeVote"]
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load Dataset


In [2]:

data = pd.read_excel(DATASET_PATH, sheet_name="data")
feature_columns = [column for column in data.columns if column not in DROP_FROM_FEATURES]

X = data[feature_columns].copy()
y = data[TARGET].astype(str)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [column for column in X.columns if column not in numeric_features]

print("Dataset:", DATASET_PATH.name)
print("Rows, columns:", data.shape)
print("Target:", TARGET)
print("Predictors:", len(feature_columns))
print("Class distribution (%):")
print((y.value_counts(normalize=True).reindex(CLASS_ORDER) * 100).round(1))


Dataset: ieq_model_dataset.xlsx
Rows, columns: (6834, 22)
Target: IEQ satisfaction 3-class
Predictors: 20
Class distribution (%):
IEQ satisfaction 3-class
dissatisfied     7.1
neutral         27.1
satisfied       65.8
Name: proportion, dtype: float64


## 3. Preprocessing, Metrics, and Search Spaces


In [3]:

def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ],
        remainder="drop",
    )


def make_pipeline(base_model) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocess", make_preprocessor()),
            ("model", base_model),
        ]
    )


def ordinal_mae(y_true: pd.Series, y_pred: np.ndarray) -> float:
    true_numbers = y_true.map(CLASS_TO_NUMBER).to_numpy()
    pred_numbers = pd.Series(y_pred).map(CLASS_TO_NUMBER).to_numpy()
    return mean_absolute_error(true_numbers, pred_numbers)


def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "ordinal_mae": float(ordinal_mae(y_true, y_pred)),
    }


def make_model_specs() -> dict[str, dict]:
    return {
        "Logistic regression": {
            "estimator": LogisticRegression(max_iter=2500, random_state=SEED),
            "params": {
                "model__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
                "model__class_weight": ["balanced", None],
            },
            "n_iter": 6,
        },
        "Random Forest": {
            "estimator": RandomForestClassifier(n_jobs=1, random_state=SEED),
            "params": {
                "model__n_estimators": [150, 300],
                "model__max_depth": [None, 12, 20],
                "model__min_samples_leaf": [2, 4, 8],
                "model__min_samples_split": [2, 10],
                "model__max_features": ["sqrt", 0.5],
                "model__class_weight": ["balanced", "balanced_subsample"],
            },
            "n_iter": 8,
        },
        "Extra Trees": {
            "estimator": ExtraTreesClassifier(n_jobs=1, random_state=SEED),
            "params": {
                "model__n_estimators": [700],
                "model__criterion": ["entropy"],
                "model__max_depth": [20],
                "model__min_samples_leaf": [2],
                "model__min_samples_split": [10],
                "model__max_features": [None],
                "model__class_weight": ["balanced"],
                "model__bootstrap": [True],
                "model__max_samples": [0.9],
            },
            "n_iter": 1,
        },
        "Gradient boosting": {
            "estimator": GradientBoostingClassifier(random_state=SEED),
            "params": {
                "model__n_estimators": [80, 120, 160],
                "model__learning_rate": [0.03, 0.05, 0.08],
                "model__max_depth": [2, 3],
                "model__min_samples_leaf": [3, 10],
                "model__subsample": [0.85, 1.0],
                "model__max_features": [None, "sqrt"],
            },
            "n_iter": 8,
        },
        "ANN / MLP": {
            "estimator": MLPClassifier(max_iter=250, early_stopping=False, random_state=SEED),
            "params": {
                "model__hidden_layer_sizes": [(16,), (32,), (32, 16)],
                "model__activation": ["relu"],
                "model__alpha": [0.001, 0.01],
                "model__learning_rate_init": [0.001, 0.003],
            },
            "n_iter": 5,
        },
    }


## 4. Tune Hyperparameters

Each model is tuned with `RandomizedSearchCV` and macro F1 scoring.


In [4]:

model_specs = make_model_specs()
search_cv = StratifiedKFold(n_splits=TUNING_SPLITS, shuffle=True, random_state=SEED)

best_params = {}
best_rows = []
cv_result_frames = []

for model_name in MODEL_ORDER:
    spec = model_specs[model_name]
    pipeline = make_pipeline(clone(spec["estimator"]))
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=spec["params"],
        n_iter=spec["n_iter"],
        scoring=SCORING,
        cv=search_cv,
        random_state=SEED,
        n_jobs=1,
        refit=True,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    print(f"\nTuning {model_name}...")
    start = time.perf_counter()
    search.fit(X, y)
    elapsed = time.perf_counter() - start

    best_params[model_name] = search.best_params_
    best_rows.append(
        {
            "model": model_name,
            "best_cv_macro_f1": search.best_score_,
            "tuning_seconds": elapsed,
            "best_params_json": json.dumps(search.best_params_, sort_keys=True),
        }
    )

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results = cv_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_fit_time", "std_fit_time", "params"]].copy()
    cv_results.insert(0, "model", model_name)
    cv_results["params_json"] = cv_results["params"].apply(lambda value: json.dumps(value, sort_keys=True))
    cv_results = cv_results.drop(columns="params")
    cv_result_frames.append(cv_results)

best_params_table = pd.DataFrame(best_rows).sort_values("best_cv_macro_f1", ascending=False)
tuning_results = pd.concat(cv_result_frames, ignore_index=True)

best_params_path = TABLE_DIR / "hyperparameter_tuning_best_params.csv"
tuning_results_path = TABLE_DIR / "hyperparameter_tuning_cv_results.csv"
best_params_table.to_csv(best_params_path, index=False)
tuning_results.to_csv(tuning_results_path, index=False)

print("\nBest tuned parameters:")
print(best_params_table[["model", "best_cv_macro_f1", "tuning_seconds"]].to_string(index=False))
print("Saved", best_params_path)
print("Saved", tuning_results_path)



Tuning Logistic regression...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Tuning Random Forest...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Tuning Extra Trees...
Fitting 5 folds for each of 1 candidates, totalling 5 fits

Tuning Gradient boosting...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Tuning ANN / MLP...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best tuned parameters:
              model  best_cv_macro_f1  tuning_seconds
        Extra Trees          0.591090       52.397584
      Random Forest          0.582349      126.983099
          ANN / MLP          0.517312      152.756606
  Gradient boosting          0.463590       50.409967
Logistic regression          0.400758        3.654460
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\tables\hyperparameter_tuning_best_params.csv
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ie

## 5. Evaluate Tuned Models

The best hyperparameters are evaluated with 5-fold cross-validation and the full metric set.


In [5]:

evaluation_cv = StratifiedKFold(n_splits=EVALUATION_SPLITS, shuffle=True, random_state=SEED)
fold_rows = []
prediction_rows = []

for model_name in MODEL_ORDER:
    print(f"\nEvaluating tuned {model_name}...")
    spec = model_specs[model_name]
    for fold, (train_index, test_index) in enumerate(evaluation_cv.split(X, y), start=1):
        pipeline = make_pipeline(clone(spec["estimator"]))
        pipeline.set_params(**best_params[model_name])
        if "model__random_state" in pipeline.get_params():
            pipeline.set_params(model__random_state=SEED + fold)

        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        start = time.perf_counter()
        pipeline.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - start

        start = time.perf_counter()
        y_pred = pipeline.predict(X_test)
        predict_seconds = time.perf_counter() - start

        row = {
            "model": model_name,
            "fold": fold,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "total_seconds": fit_seconds + predict_seconds,
        }
        row.update(evaluate_predictions(y_test, y_pred))
        fold_rows.append(row)

        for row_index, true_label, predicted_label in zip(test_index, y_test, y_pred):
            prediction_rows.append(
                {
                    "model": model_name,
                    "fold": fold,
                    "row_index": row_index,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                }
            )

fold_results = pd.DataFrame(fold_rows)
cv_predictions = pd.DataFrame(prediction_rows)
metric_columns = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "ordinal_mae", "fit_seconds", "predict_seconds", "total_seconds"]

summary = fold_results.groupby("model")[metric_columns].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary["macro_f1_per_second"] = summary["macro_f1_mean"] / summary["total_seconds_mean"]
summary = summary.merge(best_params_table, on="model", how="left")

folds_paths = (TABLE_DIR / "tuned_model_comparison_5fold_folds.csv", TABLE_DIR / "model_comparison_5fold_folds.csv")
predictions_paths = (TABLE_DIR / "tuned_model_comparison_5fold_predictions.csv", TABLE_DIR / "model_comparison_5fold_predictions.csv")
summary_paths = (TABLE_DIR / "tuned_model_comparison_5fold_summary.csv", TABLE_DIR / "model_comparison_5fold_summary.csv")
for folds_path in folds_paths:
    fold_results.to_csv(folds_path, index=False)
for predictions_path in predictions_paths:
    cv_predictions.to_csv(predictions_path, index=False)
for summary_path in summary_paths:
    summary.to_csv(summary_path, index=False)

print("\nTuned 5-fold evaluation:")
print(summary.sort_values("macro_f1_mean", ascending=False)[["model", "accuracy_mean", "balanced_accuracy_mean", "macro_f1_mean", "weighted_f1_mean", "ordinal_mae_mean", "total_seconds_mean", "best_cv_macro_f1"]].to_string(index=False))
for output_path in (*folds_paths, *predictions_paths, *summary_paths):
    print("Saved", output_path)



Evaluating tuned Logistic regression...

Evaluating tuned Random Forest...

Evaluating tuned Extra Trees...

Evaluating tuned Gradient boosting...

Evaluating tuned ANN / MLP...

Tuned 5-fold evaluation:
              model  accuracy_mean  balanced_accuracy_mean  macro_f1_mean  weighted_f1_mean  ordinal_mae_mean  total_seconds_mean  best_cv_macro_f1
        Extra Trees       0.722562                0.591758       0.590984          0.719881          0.314750            7.889927          0.591090
      Random Forest       0.714808                0.586930       0.584951          0.710669          0.324991            1.983206          0.582349
          ANN / MLP       0.693593                0.490320       0.510187          0.675240          0.343721            5.561022          0.517312
  Gradient boosting       0.690371                0.443643       0.461041          0.633868          0.358210            1.041814          0.463590
Logistic regression       0.482732                0.464

## 6. Plot Tuned Model Performance


In [6]:

def save_current_figure(fig, name: str, *aliases: str) -> None:
    for figure_name in (name, *aliases):
        output_path = FIGURE_DIR / figure_name
        save_figure(fig, output_path)
        print("Saved", output_path.with_suffix(".png"))
        print("Saved", output_path.with_suffix(".pdf"))

plot_df = summary.set_index("model").reindex(MODEL_ORDER)
performance_metrics = [
    ("accuracy_mean", "accuracy_std", "Accuracy"),
    ("balanced_accuracy_mean", "balanced_accuracy_std", "Balanced accuracy"),
    ("macro_f1_mean", "macro_f1_std", "Macro F1"),
    ("weighted_f1_mean", "weighted_f1_std", "Weighted F1"),
]
x = np.arange(len(MODEL_ORDER))
bar_colors = [MODEL_COLORS[model] for model in MODEL_ORDER]

fig, axes = plt.subplots(ncols=4, figsize=(13.6, 3.9), sharey=True)
for ax, (mean_col, std_col, label) in zip(axes, performance_metrics):
    values = plot_df[mean_col].astype(float)
    errors = plot_df[std_col].astype(float)
    bars = ax.bar(
        x,
        values,
        yerr=errors,
        capsize=2.5,
        color=bar_colors,
        edgecolor=PAPER_COLORS["border"],
        linewidth=0.55,
        error_kw={"elinewidth": 0.8, "capthick": 0.8, "ecolor": PAPER_COLORS["axis"]},
    )
    for bar, model, value in zip(bars, MODEL_ORDER, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value - 0.035,
            f"{value:.2f}",
            ha="center",
            va="top",
            fontsize=8.2,
            fontweight="bold",
            color=model_value_label_color(model),
        )
    ax.set_title(label, fontsize=12, fontweight="bold", pad=8)
    ax.set_ylim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[model] for model in MODEL_ORDER])
    style_axes(ax, grid_axis="y")

axes[0].set_ylabel("Score")
for ax in axes[1:]:
    ax.tick_params(axis="y", labelleft=False)

fig.suptitle("Tuned model performance by metric (5-fold cross-validation)", fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
save_current_figure(fig, "tuned_model_comparison_performance", "model_comparison_performance")
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.4))
runtime = plot_df["total_seconds_mean"].astype(float)
runtime_std = plot_df["total_seconds_std"].astype(float)
y_pos = np.arange(len(MODEL_ORDER))
ax.barh(
    y_pos,
    runtime,
    xerr=runtime_std,
    capsize=2.5,
    color=bar_colors,
    edgecolor=PAPER_COLORS["border"],
    linewidth=0.55,
    error_kw={"elinewidth": 0.8, "capthick": 0.8, "ecolor": PAPER_COLORS["axis"]},
)
ax.set_title("Tuned model runtime per fold", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean fit + predict time (s)")
ax.set_yticks(y_pos)
ax.set_yticklabels([MODEL_LABELS[model].replace("\n", " ") for model in MODEL_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, runtime.max() * 1.22)
style_axes(ax, grid_axis="x")
for index, value in enumerate(runtime):
    ax.text(value + runtime.max() * 0.025, index, f"{value:.2f}", va="center", fontsize=9, color=PAPER_COLORS["text"])
fig.tight_layout()
save_current_figure(fig, "tuned_model_comparison_runtime", "model_comparison_runtime")
plt.show()

fig, ax = plt.subplots(figsize=(7.0, 4.6))
for model in MODEL_ORDER:
    row = plot_df.loc[model]
    ax.errorbar(
        row["total_seconds_mean"],
        row["macro_f1_mean"],
        xerr=row["total_seconds_std"],
        yerr=row["macro_f1_std"],
        fmt="o",
        markersize=7,
        color=MODEL_COLORS[model],
        ecolor=PAPER_COLORS["axis"],
        elinewidth=0.8,
        capsize=2.5,
        markeredgecolor="white",
        markeredgewidth=0.8,
    )
    ax.annotate(
        MODEL_LABELS[model].replace("\n", " "),
        (row["total_seconds_mean"], row["macro_f1_mean"]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=PAPER_COLORS["text"],
    )
ax.set_title("Tuned performance-efficiency trade-off", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean fit + predict time per fold (s)")
ax.set_ylabel("Mean macro F1")
ax.set_xlim(0, plot_df["total_seconds_mean"].max() * 1.28)
macro_f1_min = plot_df["macro_f1_mean"].min()
macro_f1_max = plot_df["macro_f1_mean"].max()
ax.set_ylim(max(0, macro_f1_min - 0.08), min(1, macro_f1_max + 0.08))
style_axes(ax, grid_axis="both")
fig.tight_layout()
save_current_figure(fig, "tuned_model_comparison_efficiency_tradeoff", "model_comparison_efficiency_tradeoff")
plt.show()


Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_performance.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_performance.pdf
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_performance.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_performance.pdf


C:\Users\Dell Latitude 5540\AppData\Local\Temp\ipykernel_44720\1240878561.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_runtime.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_runtime.pdf
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_runtime.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_runtime.pdf


C:\Users\Dell Latitude 5540\AppData\Local\Temp\ipykernel_44720\1240878561.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_efficiency_tradeoff.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_efficiency_tradeoff.pdf
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_efficiency_tradeoff.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_efficiency_tradeoff.pdf


C:\Users\Dell Latitude 5540\AppData\Local\Temp\ipykernel_44720\1240878561.py:121: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Confusion Matrices


In [7]:

confusion_rows = []
fig, axes = plt.subplots(ncols=len(MODEL_ORDER), figsize=(13.6, 3.6), sharex=True, sharey=True)

for ax, model in zip(axes, MODEL_ORDER):
    model_predictions = cv_predictions[cv_predictions["model"] == model]
    counts = confusion_matrix(
        model_predictions["true_label"],
        model_predictions["predicted_label"],
        labels=CLASS_ORDER,
    )
    row_totals = counts.sum(axis=1, keepdims=True)
    normalized = counts / np.where(row_totals == 0, 1, row_totals) * 100

    image = ax.imshow(normalized, vmin=0, vmax=100, cmap=CONFUSION_CMAP)
    ax.set_title(MODEL_LABELS[model].replace("\n", " "), fontsize=10.5, fontweight="bold", pad=8)
    ax.set_xticks(np.arange(len(CLASS_ORDER)))
    ax.set_yticks(np.arange(len(CLASS_ORDER)))
    ax.set_xticklabels(CLASS_DISPLAY_LABELS, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_DISPLAY_LABELS)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for true_index, true_label in enumerate(CLASS_ORDER):
        for predicted_index, predicted_label in enumerate(CLASS_ORDER):
            percent = normalized[true_index, predicted_index]
            count = counts[true_index, predicted_index]
            confusion_rows.append(
                {
                    "model": model,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                    "count": int(count),
                    "row_percent": percent,
                }
            )
            ax.text(
                predicted_index,
                true_index,
                f"{percent:.0f}%\n{count}",
                ha="center",
                va="center",
                fontsize=8.2,
                color="white" if percent >= 50 else PAPER_COLORS["text"],
            )

axes[0].set_ylabel("True class")
for ax in axes:
    ax.set_xlabel("Predicted class")

fig.suptitle("Tuned out-of-fold confusion matrices (5-fold cross-validation)", fontsize=14, fontweight="bold", y=0.98)
fig.subplots_adjust(wspace=0.22, bottom=0.27, top=0.78, right=0.92)
colorbar_axis = fig.add_axes([0.935, 0.30, 0.012, 0.43])
colorbar = fig.colorbar(image, cax=colorbar_axis)
colorbar.set_label("Row-normalized share (%)")

confusion_table = pd.DataFrame(confusion_rows)
for confusion_path in (TABLE_DIR / "tuned_model_confusion_matrices.csv", TABLE_DIR / "model_confusion_matrices.csv"):
    confusion_table.to_csv(confusion_path, index=False)
    print("Saved", confusion_path)

save_current_figure(fig, "tuned_model_comparison_confusion_matrices", "model_comparison_confusion_matrices")
plt.show()


Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\tables\tuned_model_confusion_matrices.csv
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\tables\model_confusion_matrices.csv
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_confusion_matrices.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\tuned_model_comparison_confusion_matrices.pdf
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_confusion_matrices.png
Saved C:\Users\Dell Latitude 5540\OneDrive - VUT\12. semestr\ML\Prism\03_Code\ieq_paper\02_outputs\figures\model_comparison_confusion_matrices.pdf


C:\Users\Dell Latitude 5540\AppData\Local\Temp\ipykernel_44720\1109613756.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary Table


In [8]:

interpretation_columns = [
    "model",
    "accuracy_mean",
    "balanced_accuracy_mean",
    "macro_f1_mean",
    "weighted_f1_mean",
    "ordinal_mae_mean",
    "total_seconds_mean",
    "best_cv_macro_f1",
    "tuning_seconds",
]

print(summary[interpretation_columns].sort_values("macro_f1_mean", ascending=False).to_string(index=False))


              model  accuracy_mean  balanced_accuracy_mean  macro_f1_mean  weighted_f1_mean  ordinal_mae_mean  total_seconds_mean  best_cv_macro_f1  tuning_seconds
        Extra Trees       0.722562                0.591758       0.590984          0.719881          0.314750            7.889927          0.591090       52.397584
      Random Forest       0.714808                0.586930       0.584951          0.710669          0.324991            1.983206          0.582349      126.983099
          ANN / MLP       0.693593                0.490320       0.510187          0.675240          0.343721            5.561022          0.517312      152.756606
  Gradient boosting       0.690371                0.443643       0.461041          0.633868          0.358210            1.041814          0.463590       50.409967
Logistic regression       0.482732                0.464029       0.400758          0.520650          0.671057            0.207107          0.400758        3.654460
